# 04 - Water leak classifier: training and cross-validation

Trains the classifier in `.spec/features/water-leak-classifier.md`: a fine-tuned
`all-MiniLM-L6-v2` that labels a report `WATER_LEAK` or `NOT_LEAK`.

**Data.** [Civic and Municipality Complaint System dataset](https://www.kaggle.com/datasets/wajahattaj/civic-and-municipality-complaint-system-dataset)
(MIT). Download `municipal_training_set_1100.csv` into `ml-research/data/water-leak/`.
`ml_research.water_leak_data` reduces its 1,100 rows to one row per distinct sentence.

**Evaluation** is 5-fold cross-validation over distinct sentences, so a sentence is
never on both sides of a split. It can be run as often as needed, which means the
scores below are optimistic about the chosen settings. The model is evaluated only
on these US street complaints, not on what estate residents write.

Targets at the 0.80 routing threshold: precision >= 0.95 and F1 >= 0.85.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline

from ml_research.water_leak_data import WATER_LEAK, prepare
from ml_research.water_leak_training import device, predict_logits, train_model

DATA = Path("../data/water-leak/municipal_training_set_1100.csv")
ARTIFACTS = Path("../artifacts/water-leak")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.80
TARGET_PRECISION = 0.95
TARGET_F1 = 0.85
FOLDS = 5
SEED = 42

device

## Data

In [ ]:
sentences = prepare(DATA)
texts = sentences["sentence"].tolist()
labels = (sentences["label"] == WATER_LEAK).astype(int).to_numpy()

print(f"{len(sentences)} sentences, {labels.sum()} leaks, {len(labels) - labels.sum()} not")
folds = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(texts, labels))
print("leaks held out per fold:", [int(labels[test].sum()) for _, test in folds])

## Training

A plain PyTorch loop. Leaks are about 1 in 8, so the loss can be weighted by
class: `balanced`, its square root (`sqrt`), or `none`. Weighting makes the model
keener to call something a leak, which trades precision for recall. Padding is per batch; the attention mask makes it invisible to the model.

In [ ]:
def cross_validate(epochs, learning_rate, class_weight, seed=SEED):
    held_out = np.zeros((len(texts), 2), dtype=np.float32)
    for train, test in folds:
        model = train_model(
            [texts[i] for i in train], labels[train], epochs, learning_rate, class_weight, seed
        )
        held_out[test] = predict_logits(model, [texts[i] for i in test])
        del model
    return held_out

## Calibration and metrics

One temperature is fitted on the pooled held-out logits, so that a probability of
0.9 means right about 90% of the time. `metrics` scores a threshold as routing
would use it: a report counts as a predicted leak only at or above that threshold.

In [ ]:
def fit_temperature(logits, targets):
    logits = torch.tensor(logits, dtype=torch.float64)
    targets = torch.tensor(targets)
    log_temperature = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=200)

    def closure():
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(logits / log_temperature.exp(), targets)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(log_temperature.detach().exp())


def leak_probability(logits, temperature=1.0):
    scaled = torch.tensor(logits, dtype=torch.float64) / temperature
    return torch.softmax(scaled, dim=1)[:, 1].numpy()


def metrics(probability, targets, threshold=THRESHOLD):
    predicted = probability >= threshold
    true_positive = int((predicted & (targets == 1)).sum())
    false_positive = int((predicted & (targets == 0)).sum())
    false_negative = int((~predicted & (targets == 1)).sum())
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": true_positive,
        "fp": false_positive,
        "fn": false_negative,
    }


def calibration_error(probability, targets, bins=10):
    predicted = (probability >= 0.5).astype(int)
    confidence = np.maximum(probability, 1 - probability)
    edges = np.linspace(0.5, 1.0, bins + 1)
    error = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        members = (confidence > low) & (confidence <= high)
        if members.any():
            accuracy = (predicted[members] == targets[members]).mean()
            error += members.mean() * abs(accuracy - confidence[members].mean())
    return error

## Iteration log

Each row is one run of cross-validation. The TF-IDF row is a reference on the same folds,
with its own fitted temperature. It is not a candidate for shipping.

In [ ]:
CONFIGS = [
    # Round 1: learning rate and epochs, with balanced class weights.
    {"epochs": 3, "learning_rate": 5e-5, "class_weight": "balanced"},
    {"epochs": 5, "learning_rate": 5e-5, "class_weight": "balanced"},
    {"epochs": 8, "learning_rate": 5e-5, "class_weight": "balanced"},
    {"epochs": 3, "learning_rate": 1e-4, "class_weight": "balanced"},
    {"epochs": 5, "learning_rate": 1e-4, "class_weight": "balanced"},
    {"epochs": 8, "learning_rate": 1e-4, "class_weight": "balanced"},
    {"epochs": 5, "learning_rate": 2e-4, "class_weight": "balanced"},
    # Round 2: round 1 missed precision through false positives, so weaken the weighting.
    {"epochs": 3, "learning_rate": 1e-4, "class_weight": "sqrt"},
    {"epochs": 5, "learning_rate": 1e-4, "class_weight": "sqrt"},
    {"epochs": 8, "learning_rate": 1e-4, "class_weight": "sqrt"},
    {"epochs": 3, "learning_rate": 1e-4, "class_weight": "none"},
    {"epochs": 5, "learning_rate": 1e-4, "class_weight": "none"},
    {"epochs": 8, "learning_rate": 1e-4, "class_weight": "none"},
]


def summarise(name, logits):
    temperature = fit_temperature(logits, labels)
    probability = leak_probability(logits, temperature)
    at_threshold = metrics(probability, labels)
    at_half = metrics(probability, labels, 0.5)
    return {
        "run": name,
        "temperature": round(temperature, 3),
        "precision@0.80": round(at_threshold["precision"], 3),
        "recall@0.80": round(at_threshold["recall"], 3),
        "f1@0.80": round(at_threshold["f1"], 3),
        "f1@0.50": round(at_half["f1"], 3),
        "fp@0.80": at_threshold["fp"],
        "fn@0.80": at_threshold["fn"],
        "ece": round(calibration_error(probability, labels), 3),
    }


baseline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)
baseline_logits = np.zeros((len(texts), 2))
for train, test in folds:
    baseline.fit([texts[i] for i in train], labels[train])
    score = baseline.decision_function([texts[i] for i in test])
    baseline_logits[test] = np.column_stack([-score / 2, score / 2])

log = [summarise("TF-IDF reference", baseline_logits)]
runs = {}
for config in CONFIGS:
    name = f"MiniLM epochs={config['epochs']} lr={config['learning_rate']:g} weight={config['class_weight']}"
    start = time.time()
    runs[name] = (config, cross_validate(**config))
    row = summarise(name, runs[name][1])
    row["seconds"] = round(time.time() - start)
    log.append(row)
    print(row, flush=True)

iteration_log = pd.DataFrame(log)
iteration_log.to_csv(ARTIFACTS / "iteration-log.csv", index=False)
iteration_log

## Stability across seeds

With 47 leaks, one sentence moves precision by about two points, so a single run can
meet the target by luck. The three best runs are repeated with other training seeds,
and the choice is made on the average: highest mean F1 at 0.80 among runs whose mean
precision meets the target, otherwise the highest mean precision.

In [ ]:
EXTRA_SEEDS = [7, 123]
minilm = iteration_log[iteration_log["run"].str.startswith("MiniLM")]
shortlist = minilm.sort_values(["precision@0.80", "f1@0.80", "ece"], ascending=[False, False, True]).head(3)

stability = []
for name in shortlist["run"]:
    config, logits = runs[name]
    rows = [metrics(leak_probability(logits, fit_temperature(logits, labels)), labels)]
    for seed in EXTRA_SEEDS:
        repeat = cross_validate(**config, seed=seed)
        rows.append(metrics(leak_probability(repeat, fit_temperature(repeat, labels)), labels))
    stability.append(
        {
            "run": name,
            "mean_precision": round(np.mean([r["precision"] for r in rows]), 3),
            "min_precision": round(min(r["precision"] for r in rows), 3),
            "mean_recall": round(np.mean([r["recall"] for r in rows]), 3),
            "mean_f1": round(np.mean([r["f1"] for r in rows]), 3),
            "fp_per_seed": [r["fp"] for r in rows],
            "fn_per_seed": [r["fn"] for r in rows],
        }
    )
stability = pd.DataFrame(stability)
stability.to_csv(ARTIFACTS / "stability.csv", index=False)
display(stability)

meeting = stability[stability["mean_precision"] >= TARGET_PRECISION]
ranked = (
    meeting.sort_values("mean_f1", ascending=False)
    if not meeting.empty
    else stability.sort_values(["mean_precision", "mean_f1"], ascending=False)
)
chosen_name = ranked.iloc[0]["run"]
chosen_config, chosen_logits = runs[chosen_name]
temperature = fit_temperature(chosen_logits, labels)
probability = leak_probability(chosen_logits, temperature)
result = metrics(probability, labels)
print(chosen_name, "temperature", round(temperature, 3))
print(result)
print("targets met by the mean:", ranked.iloc[0]["mean_precision"] >= TARGET_PRECISION and ranked.iloc[0]["mean_f1"] >= TARGET_F1)

## Mistakes and lookalikes

What a person should read before trusting the numbers.

In [ ]:
pd.set_option("display.max_colwidth", 130, "display.width", 200, "display.max_rows", 100)
scored = sentences.assign(p_leak=probability.round(3))[["sentence", "label", "p_leak"]]
is_leak = scored["label"] == WATER_LEAK

print("False positives at 0.80 (a contractor would be emailed for nothing)")
display(scored[~is_leak & (scored["p_leak"] >= THRESHOLD)].sort_values("p_leak", ascending=False))
print("Leaks that would NOT route automatically")
display(scored[is_leak & (scored["p_leak"] < THRESHOLD)].sort_values("p_leak"))
print("Non-leaks the model found most leak-like")
display(scored[~is_leak].sort_values("p_leak", ascending=False).head(12))

## Threshold and calibration

In [ ]:
thresholds = np.arange(0.50, 0.96, 0.05)
curve = pd.DataFrame(
    [{"threshold": round(float(t), 2), **metrics(probability, labels, t)} for t in thresholds]
).round(3)
display(curve)

confidence = np.maximum(probability, 1 - probability)
predicted = (probability >= 0.5).astype(int)
edges = np.linspace(0.5, 1.0, 6)
points = [
    (confidence[(confidence > low) & (confidence <= high)].mean(),
     (predicted == labels)[(confidence > low) & (confidence <= high)].mean())
    for low, high in zip(edges[:-1], edges[1:])
    if ((confidence > low) & (confidence <= high)).any()
]
figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(curve["threshold"], curve["precision"], marker="o", label="precision")
axes[0].plot(curve["threshold"], curve["recall"], marker="o", label="recall (share routed)")
axes[0].axvline(THRESHOLD, color="grey", linestyle="--")
axes[0].set_xlabel("threshold")
axes[0].legend()
axes[1].plot([0.5, 1], [0.5, 1], color="grey", linestyle="--")
axes[1].plot(*zip(*points), marker="o")
axes[1].set_xlabel("confidence")
axes[1].set_ylabel("accuracy")
axes[1].set_title("Reliability")
figure.tight_layout()

In [ ]:
(ARTIFACTS / "check-1.json").write_text(
    json.dumps(
        {
            "run": chosen_name,
            "config": chosen_config,
            "temperature": temperature,
            "threshold": THRESHOLD,
            "folds": FOLDS,
            "seed": SEED,
            "sentences": len(texts),
            "leaks": int(labels.sum()),
            "metrics": result,
            "stability": stability[stability["run"] == chosen_name].iloc[0].to_dict(),
            "ece": calibration_error(probability, labels),
        },
        indent=2,
    )
)

## Results (2026-11-09)

Outputs are cleared before committing, so the results of the recorded run are kept here.
Seed 42 unless stated. Scores are at the 0.80 threshold, after calibration.

| Run | Precision | Recall | F1 | FP | FN |
| --- | --- | --- | --- | --- | --- |
| TF-IDF reference | 0.978 | 0.936 | 0.957 | 1 | 3 |
| epochs=3 lr=5e-5 balanced | 0.849 | 0.957 | 0.900 | 8 | 2 |
| epochs=5 lr=5e-5 balanced | 0.900 | 0.957 | 0.928 | 5 | 2 |
| epochs=8 lr=5e-5 balanced | 0.900 | 0.957 | 0.928 | 5 | 2 |
| epochs=3 lr=1e-4 balanced | 0.938 | 0.957 | 0.947 | 3 | 2 |
| epochs=5 lr=1e-4 balanced | 0.938 | 0.957 | 0.947 | 3 | 2 |
| epochs=8 lr=1e-4 balanced | 0.938 | 0.957 | 0.947 | 3 | 2 |
| epochs=5 lr=2e-4 balanced | 0.900 | 0.957 | 0.928 | 5 | 2 |
| epochs=3 lr=1e-4 sqrt | 0.918 | 0.957 | 0.938 | 4 | 2 |
| epochs=5 lr=1e-4 sqrt | 0.900 | 0.957 | 0.928 | 5 | 2 |
| epochs=8 lr=1e-4 sqrt | 0.938 | 0.957 | 0.947 | 3 | 2 |
| epochs=3 lr=1e-4 none | 0.978 | 0.936 | 0.957 | 1 | 3 |
| epochs=5 lr=1e-4 none | 0.918 | 0.957 | 0.938 | 4 | 2 |
| epochs=8 lr=1e-4 none | 0.978 | 0.957 | 0.968 | 1 | 2 |

Repeated with seeds 42, 7 and 123:

| Run | Mean precision | Lowest precision | Mean recall | Mean F1 | FP per seed |
| --- | --- | --- | --- | --- | --- |
| epochs=8 lr=1e-4 none | 0.926 | 0.880 | 0.957 | 0.941 | 1, 4, 6 |
| epochs=3 lr=1e-4 none | 0.944 | 0.900 | 0.929 | 0.936 | 1, 5, 2 |
| **epochs=8 lr=1e-4 sqrt (chosen)** | **0.951** | 0.938 | 0.957 | 0.954 | 3, 1, 3 |

The best single run (epochs=8, none) did not hold up across seeds. The chosen run
meets both targets on average, by a margin of about one sentence. TF-IDF matches it
on these sentences.